# Baseline - Standard Fine-Tuning (Condition 1)
## What this notebook does

This is the reference condition - standard supervised fine-tuning with no fairness intervention.

| Property | Value |
|---|---|
| Training data | Original (no CDA) |
| Trainable params | All (~110M BERT / ~66M DistilBERT) |
| Fairness loss | None |
| Class weights | From `data/class_weights.json` (shared) |

## 1. Install packages

In [ ]:
# Install PyTorch with CUDA 12.1 support for RTX 4060 Ti
%pip install torch torchvision --index-url https://download.pytorch.org/whl/cu121 --quiet
%pip install transformers==4.44.0 datasets==2.20.0 scikit-learn pandas numpy==1.26.4 codecarbon scipy --quiet

## 2. Imports

In [ ]:
import os
import json
import time
import random
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.special import softmax

from datasets import Dataset, DatasetDict
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    roc_auc_score,
)

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    DataCollatorWithPadding,
    TrainingArguments,
    Trainer,
    set_seed,
)

from codecarbon import EmissionsTracker

## 3. Configuration

In [ ]:
# Model
MODEL_NAME = "bert-base-uncased"
#MODEL_NAME = "distilbert-base-uncased"
#MODEL_NAME = "roberta-base"

# Run mode
RUN_MODE = "STANDARD"   # "DEBUG" | "STANDARD" | "FULL"

if RUN_MODE == "DEBUG":
    MAX_LENGTH    = 64
    EPOCHS        = 1
    TRAIN_BATCH   = 4
    EVAL_BATCH    = 4
    SAVE_STRATEGY = "no"
    LOAD_BEST     = False
elif RUN_MODE == "STANDARD":
    MAX_LENGTH    = 128
    EPOCHS        = 3
    TRAIN_BATCH   = 16
    EVAL_BATCH    = 32
    SAVE_STRATEGY = "no"
    LOAD_BEST     = False
# Paths
BASE_DIR    = r"C:\Users\scanc\Desktop\v3"
DATA_DIR    = os.path.join(BASE_DIR, "data")
RESULTS_DIR = os.path.join(BASE_DIR, "baseline_results")

os.makedirs(RESULTS_DIR, exist_ok=True)

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

safe_model_name = MODEL_NAME.replace("/", "_").replace("-", "_")

print(f"Model       : {MODEL_NAME}")
print(f"Run mode    : {RUN_MODE}")
print(f"Max length  : {MAX_LENGTH}")
print(f"Epochs      : {EPOCHS}")
print(f"Data dir    : {DATA_DIR}")
print(f"Results dir : {RESULTS_DIR}")

## 4. Check device

In [ ]:
print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    device = "cuda"
    print("GPU :", torch.cuda.get_device_name(0))
    print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3,1), "GB")
elif torch.backends.mps.is_available():
    device = "mps"
    print("Using Apple MPS")
else:
    device = "cpu"
    print("No GPU - CPU only")
print("Active device:", device)

## 5. Load shared splits and class weights

All data comes from `llmthesis_data_preparation.ipynb`.
Baseline loads `train_raw.csv` - no CDA augmentation.

In [ ]:
# Load splits
train_df = pd.read_csv(os.path.join(DATA_DIR, "train_raw.csv"))
val_df   = pd.read_csv(os.path.join(DATA_DIR, "val.csv"))
test_df  = pd.read_csv(os.path.join(DATA_DIR, "test.csv"))

# Load shared class weights computed from the original training split.
with open(os.path.join(DATA_DIR, "class_weights.json")) as f:
    cw = json.load(f)
# Convert string keys to int
class_weights_dict = {int(k): v for k, v in cw.items()}

print(f"Train : {len(train_df):,}")
print(f"Val   : {len(val_df):,}")
print(f"Test  : {len(test_df):,}")
print(f"\nClass weights: {class_weights_dict}")
print(f"Label distribution (train):")
print(train_df["label"].value_counts(normalize=True).round(3))

# Apply DEBUG size limit if needed
if RUN_MODE == "DEBUG":
    train_df = train_df.sample(n=min(1400, len(train_df)), random_state=SEED).reset_index(drop=True)
    val_df   = val_df.sample(n=min(200, len(val_df)),   random_state=SEED).reset_index(drop=True)
    test_df  = test_df.sample(n=min(400, len(test_df)),  random_state=SEED).reset_index(drop=True)
    print(f"\nDEBUG mode - reduced to: train={len(train_df)}, val={len(val_df)}, test={len(test_df)}")

## 6. Tokenize

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def preprocess(examples):
    return tokenizer(
        examples["comment_text"],
        truncation=True,
        max_length=MAX_LENGTH,
    )

train_hf = Dataset.from_pandas(train_df[["comment_text","label"]].reset_index(drop=True))
val_hf   = Dataset.from_pandas(val_df[["comment_text","label"]].reset_index(drop=True))
test_hf  = Dataset.from_pandas(test_df[["comment_text","label"]].reset_index(drop=True))

hf_ds    = DatasetDict({"train": train_hf, "validation": val_hf, "test": test_hf})
tokenized = hf_ds.map(preprocess, batched=True)

keep = ["input_ids", "attention_mask", "token_type_ids", "label"]
for split in tokenized:
    to_remove = [c for c in tokenized[split].column_names if c not in keep]
    tokenized[split] = tokenized[split].remove_columns(to_remove)
tokenized.set_format("torch")

print("Tokenized:")
print(tokenized)

## 7. Load model

In [ ]:
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Model            : {MODEL_NAME}")
print(f"Trainable params : {trainable:,}")
print(f"Total params     : {total:,}")

## 8. Weighted Trainer

Class weights loaded from `data/class_weights.json` - computed once in the
data preparation notebook from the augmented training set. Using the same
weights across all conditions ensures class imbalance handling is not a
confounding variable.

In [ ]:
class WeightedTrainer(Trainer):
    """Trainer that applies fixed inverse-frequency class weights.

    The weights are computed once from the original training split in the data
    preparation notebook and reused across every condition, so class-imbalance
    handling is not a confounding variable.
    """

    def __init__(self, class_weights_dict, **kwargs):
        super().__init__(**kwargs)
        self.class_weights = torch.tensor(
            [class_weights_dict[0], class_weights_dict[1]],
            dtype=torch.float32,
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits

        weights = self.class_weights.to(device=logits.device, dtype=logits.dtype)
        loss = nn.functional.cross_entropy(
            logits, labels.to(logits.device), weight=weights
        )
        return (loss, outputs) if return_outputs else loss


print(f"Weight class 0 (non-toxic): {class_weights_dict[0]:.4f}")
print(f"Weight class 1 (toxic): {class_weights_dict[1]:.4f}")

## 9. Metrics

In [ ]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    if isinstance(logits, tuple):
        logits = logits[0]
    logits = np.array(logits, dtype=np.float32)

    predictions  = np.argmax(logits, axis=1)
    probabilities = softmax(logits, axis=1)[:, 1]

    accuracy                = accuracy_score(labels, predictions)
    p_bin, r_bin, f1_bin, _ = precision_recall_fscore_support(
        labels, predictions, average="binary",   zero_division=0)
    _, _, f1_w, _           = precision_recall_fscore_support(
        labels, predictions, average="weighted", zero_division=0)
    try:
        auc = roc_auc_score(labels, probabilities)
    except ValueError:
        auc = float("nan")

    return {
        "accuracy"    : round(accuracy, 4),
        "precision"   : round(p_bin,    4),
        "recall"      : round(r_bin,    4),
        "f1_binary"   : round(f1_bin,   4),
        "f1_weighted" : round(f1_w,     4),
        "auc_roc"     : round(auc,      4),
    }

## 10. Output folders

In [ ]:
output_dir  = os.path.join(RESULTS_DIR, f"{safe_model_name}_baseline_model")
logging_dir = os.path.join(RESULTS_DIR, f"{safe_model_name}_baseline_logs")
os.makedirs(output_dir,  exist_ok=True)
os.makedirs(logging_dir, exist_ok=True)
print("Output dir :", output_dir)

## 11. Training arguments

In [ ]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

training_args = TrainingArguments(
    output_dir    = output_dir,
    eval_strategy = "epoch",
    save_strategy = SAVE_STRATEGY,

    learning_rate               = 2e-5,
    per_device_train_batch_size = TRAIN_BATCH,
    per_device_eval_batch_size  = EVAL_BATCH,
    num_train_epochs            = EPOCHS,
    weight_decay                = 0.01,

    logging_dir   = logging_dir,
    logging_steps = 10,

    load_best_model_at_end = LOAD_BEST,
    metric_for_best_model  = "f1_binary" if LOAD_BEST else None,
    greater_is_better      = True,

    report_to = "none",
    seed      = SEED,
)

## 12. Train with CodeCarbon tracking

In [ ]:
tracker = EmissionsTracker(
    project_name = f"{safe_model_name}_baseline",
    output_dir   = RESULTS_DIR,
    output_file  = f"{safe_model_name}_baseline_emissions.csv",
    log_level    = "warning",
    save_to_file = True,
)

trainer = WeightedTrainer(
    class_weights_dict = class_weights_dict,
    model              = model,
    args               = training_args,
    train_dataset      = tokenized["train"],
    eval_dataset       = tokenized["validation"],
    compute_metrics    = compute_metrics,
    data_collator      = data_collator,
)

tracker.start()
wall_start = time.time()

trainer.train()

wall_end        = time.time()
emissions_kg    = tracker.stop()
runtime_seconds = wall_end - wall_start

print(f"\nTraining complete.")
print(f"  Wall-clock time : {runtime_seconds:.1f} s  ({runtime_seconds/60:.2f} min)")
print(f"  CO2 emitted     : {emissions_kg:.6f} kg CO2eq")

## 13. Final evaluation on test set

In [ ]:
results = trainer.evaluate(eval_dataset=tokenized["test"])
print("\nTest set results:")
for k, v in results.items():
    print(f"  {k:<30} {v}")

## 14. Save performance results

In [ ]:
results_df = pd.DataFrame([results])
results_df["model"]          = MODEL_NAME
results_df["method"]         = "baseline"
results_df["run_mode"]       = RUN_MODE
results_df["train_size"]     = len(train_df)
results_df["val_size"]       = len(val_df)
results_df["test_size"]      = len(test_df)
results_df["cda"]            = False
results_df["fairness_loss"]  = False
results_df["max_length"]     = MAX_LENGTH
results_df["epochs"]         = EPOCHS
results_df["runtime_seconds"]= runtime_seconds
results_df["runtime_minutes"]= runtime_seconds / 60

results_path = os.path.join(RESULTS_DIR, f"{safe_model_name}_baseline_results.csv")
results_df.to_csv(results_path, index=False)
print("Saved:", results_path)
results_df

## 15. Save predictions

Includes identity columns from `test_df` for fairness evaluation.

In [ ]:
pred_output   = trainer.predict(tokenized["test"])
logits        = pred_output.predictions
if isinstance(logits, tuple):
    logits = logits[0]
logits        = np.array(logits, dtype=np.float32)
probabilities = softmax(logits, axis=1)[:, 1]
predictions   = (probabilities >= 0.5).astype(int)
true_labels   = pred_output.label_ids

prediction_df = test_df.reset_index(drop=True).copy()
prediction_df["true_label"]     = true_labels
prediction_df["toxicity_score"] = probabilities
prediction_df["prediction"]     = predictions
prediction_df["model"]          = MODEL_NAME
prediction_df["method"]         = "baseline"

predictions_path = os.path.join(RESULTS_DIR, f"{safe_model_name}_baseline_predictions.csv")
prediction_df.to_csv(predictions_path, index=False)
print("Saved:", predictions_path)
print(f"Rows : {len(prediction_df):,}")
prediction_df.head()

## 16. Save sustainability metrics

In [ ]:
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
total_params     = sum(p.numel() for p in model.parameters())

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1024**3 if torch.cuda.is_available() else None

emissions_csv = os.path.join(RESULTS_DIR, f"{safe_model_name}_baseline_emissions.csv")
try:
    cc_df      = pd.read_csv(emissions_csv)
    energy_kwh = cc_df["energy_consumed"].iloc[-1]
except Exception:
    energy_kwh = None
    print("Could not read CodeCarbon CSV")

sustainability_df = pd.DataFrame([{
    "model"                 : MODEL_NAME,
    "method"                : "baseline",
    "cda"                   : False,
    "fairness_loss"         : False,
    "run_mode"              : RUN_MODE,
    "train_size"            : len(train_df),
    "val_size"              : len(val_df),
    "test_size"             : len(test_df),
    "runtime_seconds"       : round(runtime_seconds, 2),
    "runtime_minutes"       : round(runtime_seconds / 60, 4),
    "energy_kwh"            : energy_kwh,
    "emissions_kg_co2eq"    : round(emissions_kg, 8) if emissions_kg else None,
    "trainable_parameters"  : trainable_params,
    "total_parameters"      : total_params,
    "trainable_pct"         : round(100 * trainable_params / total_params, 4),
    "peak_gpu_memory_gb"    : round(peak_gpu_gb, 4) if peak_gpu_gb else None,
    "max_length"            : MAX_LENGTH,
    "epochs"                : EPOCHS,
}])

path = os.path.join(RESULTS_DIR, f"{safe_model_name}_baseline_sustainability.csv")
sustainability_df.to_csv(path, index=False)
print("Saved:", path)
sustainability_df